# Load silver.sales

Consolidates three Bronze sales tables into a single monthly-grain Silver staging table.

Sources:
- `bronze.sales_arancione` — CSV, monthly grain, EU/EUR
- `bronze.sales_celeste`   — CSV, daily grain (rolled up to monthly), UK/GBP
- `bronze.sales_verde`     — JSON, monthly grain, EU/EUR

`product_no` is resolved by LEFT JOIN to `silver.dim_product` on title + vintage + score.
Unmatched rows receive `product_no = '-1'` and are flagged in the output.

Write strategy: INSERT OVERWRITE — atomically replaces all Silver sales data on every run.

Starting the notebook with %run "/Workspace/Shared/notebook_init" loads common variables
and constants: CATALOG, BRONZE, SILVER, GOLD, AUDIT, RAW_FILES, PIPELINE_RUN_ID,
STATUS_RUNNING/SUCCEEDED/FAILED/NO_FILES, Utils, F, time, datetime.


In [0]:
%run "/Workspace/Shared/notebook_init"

In [0]:
%skip
%sql
truncate table vinoworld.silver.sales

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
import time

%load_ext autoreload
%autoreload 2

TARGET_TABLE     = f"{SILVER}.sales"
SOURCE_ARANCIONE = f"{BRONZE}.sales_arancione"
SOURCE_CELESTE   = f"{BRONZE}.sales_celeste"
SOURCE_VERDE     = f"{BRONZE}.sales_verde"
DIM_PRODUCT      = f"{SILVER}.dim_product"

In [0]:
nb = Utils.get_notebook_context(dbutils)
notebook_folder = nb['notebook_folder']
notebook_name   = nb['notebook_name']

step_log_id       = str(uuid.uuid4())
pipeline_run_id   = PIPELINE_RUN_ID
step_sequence     = 1
layer             = "silver"
target_table      = TARGET_TABLE
status            = STATUS_RUNNING
started_timestamp = datetime.now(timezone.utc)
rows_read         = 0
rows_written      = 0
error_message     = None

pipeline_step_log_upsert(
    spark, step_log_id, pipeline_run_id, step_sequence,
    notebook_folder, notebook_name, status, started_timestamp,
    layer, target_table
)

In [0]:
# ── Stage all three sources as temporary views ───────────────────────────────
#
# Arancione: cast columns, no structural changes (monthly grain).
#            row_hash / run_id / source_inserted_ts passed through for lineage.
# Celeste:   cast + GROUP BY rollup — daily transactions → monthly totals.
#            list_price and score are GROUP BY keys so price-change rows within
#            a month remain distinct rather than being silently collapsed.
#            row_hash is recomputed on the rolled-up keys; MAX() used for run_id
#            and source_inserted_ts since multiple Bronze rows collapse to one.
# Verde:     rename columns to align with the common Silver schema.
#            row_hash / run_id / source_inserted_ts passed through for lineage.
# ─────────────────────────────────────────────────────────────────────────────

try:
    spark.sql(f"""
        CREATE OR REPLACE TEMPORARY VIEW vw_arancione_staged AS
        SELECT
            online_retailer,
            sales_month,
            title,
            TRY_CAST(vintage  AS INT)                                AS vintage,
            variety,
            TRY_CAST(score    AS INT)                                AS score,
            CAST(TRY_CAST(list_price AS DECIMAL(10,2)) AS INT)  AS list_price,
            CAST(quantity AS INT)                                AS quantity,
            row_hash,
            inserted_ts                                          AS source_inserted_ts,
            run_id
        FROM {SOURCE_ARANCIONE}
    """)

    spark.sql(f"""
        CREATE OR REPLACE TEMPORARY VIEW vw_celeste_staged AS
        SELECT
            online_retailer,
            sales_month,
            title,
            TRY_CAST(vintage  AS INT)                                AS vintage,
            variety,
            TRY_CAST(score    AS INT)                                AS score,
            CAST(TRY_CAST(list_price AS DECIMAL(10,2)) AS INT)  AS list_price,
            SUM(TRY_CAST(quantity AS INT))                           AS quantity,
            md5(concat_ws('|', online_retailer, sales_month,
                title, vintage, variety, score, list_price))     AS row_hash,
            MAX(inserted_ts)                                     AS source_inserted_ts,
            MAX(run_id)                                          AS run_id
        FROM {SOURCE_CELESTE}
        WHERE sales_month IS NOT NULL
        GROUP BY
            online_retailer, sales_month, title, vintage,
            variety, score, list_price
    """)

    spark.sql(f"""
        CREATE OR REPLACE TEMPORARY VIEW vw_verde_staged AS
        SELECT
            store_name                                           AS online_retailer,
            year_month                                           AS sales_month,
            product                                              AS title,
            CAST(vintage    AS INT)                              AS vintage,
            variety,
            TRY_CAST(score      AS INT)                              AS score,
            CAST(TRY_CAST(sales_price AS DECIMAL(10,2)) AS INT) AS list_price,
            TRY_CAST(sales_qty  AS INT)                              AS quantity,
            row_hash,
            inserted_ts                                          AS source_inserted_ts,
            run_id
        FROM {SOURCE_VERDE}
    """)

    print("Staging views created: vw_arancione_staged, vw_celeste_staged, vw_verde_staged")

except dbutils.NotebookExit:
    raise

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = (
        f"{err['error_type']}: {err['error_message']}\n\n"
        f"{err['error_traceback']}"
    )
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )
    raise

In [0]:
# ── INSERT OVERWRITE — atomic full replace ───────────────────────────────────
#
# INSERT OVERWRITE replaces all rows in a single Delta transaction.
# Readers see either the previous complete dataset or the new one — never empty.
# product_no is resolved via LEFT JOIN to dim_product (IsRowCurrent = TRUE).
# Unmatched rows receive product_no = '-1' and are flagged in the output.
# row_hash / run_id / source_inserted_ts carried from Bronze for audit lineage.
# ─────────────────────────────────────────────────────────────────────────────

try:
    spark.sql(f"""
        INSERT OVERWRITE {TARGET_TABLE}
        SELECT
            COALESCE(dp.ProductNo, '-1')  AS product_no,
            u.online_retailer,
            u.sales_month,
            u.sales_territory,
            u.sales_currency,
            u.title,
            u.vintage,
            u.variety,
            u.score,
            u.list_price,
            u.quantity,
            u.row_hash,
            u.source_inserted_ts,
            u.run_id,
            CURRENT_TIMESTAMP()           AS inserted_ts,
            CURRENT_TIMESTAMP()           AS updated_ts
        FROM (
            SELECT online_retailer, sales_month, 'EU'  AS sales_territory, 'EUR' AS sales_currency,
                   title, vintage, variety, score, list_price, quantity,
                   row_hash, source_inserted_ts, run_id
            FROM vw_arancione_staged
            UNION ALL
            SELECT online_retailer, sales_month, 'UK'  AS sales_territory, 'GBP' AS sales_currency,
                   title, vintage, variety, score, list_price, quantity,
                   row_hash, source_inserted_ts, run_id
            FROM vw_celeste_staged
            UNION ALL
            SELECT online_retailer, sales_month, 'EU'  AS sales_territory, 'EUR' AS sales_currency,
                   title, vintage, variety, score, list_price, quantity,
                   row_hash, source_inserted_ts, run_id
            FROM vw_verde_staged
        ) u
        LEFT JOIN {DIM_PRODUCT} dp
            ON  dp.ProductName  = u.title
            AND dp.Vintage      = u.vintage
            AND dp.Score        = u.score
            AND dp.IsRowCurrent = TRUE
    """)

    rows_written = spark.table(TARGET_TABLE).count()

    if rows_written == 0:
        raise AssertionError(
            f"[{TARGET_TABLE}] INSERT OVERWRITE produced 0 rows — "
            f"check that source Bronze tables are populated."
        )

    unmatched_count = spark.sql(
        f"SELECT COUNT(*) FROM {TARGET_TABLE} WHERE product_no = '-1'"
    ).collect()[0][0]

    if unmatched_count > 0:
        print(
            f"[WARNING] [{TARGET_TABLE}] {unmatched_count:,} of {rows_written:,} rows "
            f"have no dim_product match (product_no = '-1'). "
            f"Check title / vintage / score alignment between sales and dim_product."
        )

    print(f"[{TARGET_TABLE}] {rows_written:,} rows written ({unmatched_count:,} unmatched).")

    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_SUCCEEDED

    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )

except dbutils.NotebookExit:
    raise

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = (
        f"{err['error_type']}: {err['error_message']}\n\n"
        f"{err['error_traceback']}"
    )
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )
    raise

In [0]:
%skip
%sql
-- Row summary by store, territory, and match status
SELECT
    online_retailer,
    sales_territory,
    sales_currency,
    CASE WHEN product_no = '-1' THEN 'unmatched' ELSE 'matched' END AS match_status,
    COUNT(*)       AS row_count,
    SUM(quantity)  AS total_qty
FROM vinoworld.silver.sales
GROUP BY online_retailer, sales_territory, sales_currency, match_status
ORDER BY online_retailer, match_status

In [0]:
%sql

select * from vinoworld.silver.sales
where product_no = -1

In [0]:
%skip
%sql
SELECT COUNT(*) FROM vinoworld.silver.sales

In [0]:
%skip
%sql
SELECT
    product_no,
    COUNT(DISTINCT online_retailer) AS store_count
FROM vinoworld.silver.sales
GROUP BY product_no
HAVING COUNT(DISTINCT online_retailer) > 1;